In [ ]:
dataset_url = ""
dataset_base64 = ""
n_clusters = 3
max_clusters = 10

In [ ]:
import pandas as pd
import numpy as np
import json, io, base64, warnings
warnings.filterwarnings('ignore')
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

result = {}
try:
    # ── Load data ──────────────────────────────────────────────────────────────
    if dataset_url:
        ext = dataset_url.split('.')[-1].lower()
        df = (pd.read_excel(dataset_url) if ext in ['xlsx', 'xls']
              else pd.read_json(dataset_url) if ext == 'json'
              else pd.read_csv(dataset_url))
    elif dataset_base64:
        raw = base64.b64decode(dataset_base64)
        try:
            df = pd.read_csv(io.BytesIO(raw))
        except Exception:
            df = pd.read_excel(io.BytesIO(raw))
    else:
        from sklearn.datasets import make_blobs
        X_demo, _ = make_blobs(n_samples=200, centers=3, random_state=42)
        df = pd.DataFrame(X_demo, columns=['feature_1', 'feature_2'])

    # ── Keep numeric only ──────────────────────────────────────────────────────
    df_num = df.select_dtypes(include=[np.number]).copy()
    for col in df_num.columns:
        if df_num[col].dtype in ['float64', 'int64', 'float32', 'int32']:
            df_num[col] = df_num[col].fillna(df_num[col].median())
        else:
            df_num[col] = df_num[col].fillna(df_num[col].mode()[0] if not df_num[col].mode().empty else 0)
    df_num = df_num.dropna()
    if df_num.shape[1] < 2:
        raise ValueError('Need at least 2 numeric columns for clustering')

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df_num)

    # ── Elbow method ───────────────────────────────────────────────────────────
    k_max = min(int(max_clusters), len(X_scaled) - 1, 15)
    elbow_data = []
    for k in range(2, k_max + 1):
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        km.fit(X_scaled)
        elbow_data.append({'k': k, 'inertia': round(float(km.inertia_), 4)})

    # ── Final clustering ───────────────────────────────────────────────────────
    k = int(n_clusters)
    km_final = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km_final.fit_predict(X_scaled)

    sil = float(silhouette_score(X_scaled, labels)) if k > 1 and len(set(labels)) > 1 else 0.0

    # ── PCA 2D for plot ────────────────────────────────────────────────────────
    pca = PCA(n_components=2)
    X_2d = pca.fit_transform(X_scaled)
    n_plot = min(200, len(X_2d))
    cluster_plot = [
        {'x': round(float(X_2d[i, 0]), 4), 'y': round(float(X_2d[i, 1]), 4), 'cluster': int(labels[i])}
        for i in range(n_plot)
    ]

    unique, counts = np.unique(labels, return_counts=True)
    cluster_sizes = [{'cluster': int(c), 'count': int(cnt)} for c, cnt in zip(unique, counts)]

    result = {
        'algorithm': 'kmeans',
        'n_clusters': k,
        'metrics': {
            'inertia': round(float(km_final.inertia_), 4),
            'silhouette_score': round(sil, 4),
        },
        'elbow_data': elbow_data,
        'cluster_plot': cluster_plot,
        'cluster_sizes': cluster_sizes,
    }

except Exception as e:
    import traceback
    result = {'error': str(e), 'traceback': traceback.format_exc()}

with open('/tmp/ml_result.json', 'w') as f:
    json.dump(result, f)
print(json.dumps(result, indent=2))